# Multi-dataset ScDataLoader example

`scdata` is a single-cell data **storage, compression, and loading** toolkit with
a Rust core and a Python interface. This notebook walks through the full
**path-list workflow** end to end, on several datasets at once:

1. Start from a list of dataset paths (`.zarr.zip` stores, or AnnData-readable
   sources you convert first).
2. Optionally convert `.h5ad` / `.loom` / `.mtx` / `.csv` / zarr-dir inputs into
   same-name `.zarr.zip` stores with `AnnDataZarrZipConverter`.
3. Parse every store with `launch(path)` and bulk-register them with
   `bank.register(...)` — one `DatasetId` per store.
4. Build a torch-style dataset whose samples are `(file_id, cell_id)` pairs.
5. Wrap it in `ScDataLoader`, which prefetches decoded cell batches from
   `ScDataBank` behind torch's normal sampling/shuffle/batch-size behavior.
6. Collate the decoded `CellBatch` objects into ordinary torch tensors for
   model code.

The notebook is **self-contained**: step 1 synthesizes two tiny datasets so the
rest actually runs. Swap those paths for your real stores and everything below
works unchanged.

## Concept map

```
 .zarr.zip stores ──launch()──▶ Dataset metadata ──register()──▶ ScDataBank
                                                                       │
 torch sampler yields (file_id, cell_id)                              │
         │                                                            │
         ▼                                                            ▼
  ScDataLoader ──groups each batch by file_id──▶ bank.prefetch() ──▶ CellBatch
         │                                                            │
         └──────────────────── collate_fn ◀──────────────────────────┘
                            (your code)
                                │
                                ▼
                      torch.Tensor + source ids
```

Key idea: **torch owns sampling, scdata owns decoding.** The dataset you hand to
`ScDataLoader` only needs to translate a sample index into a
`(file_id, cell_id)` pair — the bank decodes the actual expression vectors
asynchronously and hands decoded `CellBatch` objects to your `collate_fn`.

## Imports and environment

`scdata` ships a Rust core (`scdata._scdata`). The pure-Python data carriers
(`CellBatch`, `CellData`, …) work without it, but `ScDataBank` / `ScDataLoader`
need the compiled extension. `anndata` and `scipy` are only used to synthesize
inputs in this notebook — they are *not* runtime dependencies of scdata.

In [ ]:
from __future__ import annotations

from bisect import bisect_right
from collections.abc import Sequence
from pathlib import Path

import anndata as ad
import numpy as np
import torch
from scipy import sparse

from scdata import (
    AnnDataZarrZipConverter,
    DataBankConfig,
    ScheduledPrefetchConfig,
    ScDataBank,
    ScDataLoader,
    __version__,
    kernel_name,
    kernel_version,
    launch,
)

print("scdata", __version__, "/", kernel_name(), kernel_version())
print("torch", torch.__version__)

## Step 1 — Prepare input data

`store_paths` must point to scdata-readable zarr v3 stores — a `.zarr` directory
or a `.zarr.zip` archive. If your inputs are already in that form, skip this
step and set `raw_paths` to them.

To make this notebook runnable anywhere, we synthesize **two small datasets**
with `anndata` and write them as `.h5ad`:

* both share the **same 200 genes** (so mixed-file batches align column-wise),
* `sample_a` is **sparse** (CSR) with 1500 cells,
* `sample_b` is **dense** with 800 cells.

A shared gene set is what lets a single batch mix cells from both files without
reordering columns.

In [ ]:
# Resolve the repo root whether the notebook is run from / or examples/.
ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent

DATA_DIR = ROOT / "outputs" / "_nb_synthetic"
DATA_DIR.mkdir(parents=True, exist_ok=True)

N_GENES = 200


def make_adata(n_cells: int, seed: int, sparse_x: bool) -> ad.AnnData:
    rng = np.random.default_rng(seed)
    if sparse_x:
        X = sparse.random(
            n_cells, N_GENES, density=0.05, dtype=np.float32, random_state=rng
        ).tocsr()
        X.data = np.round(X.data * 100).astype(np.float32)
    else:
        X = (rng.random((n_cells, N_GENES), dtype=np.float32) * 10).astype(np.float32)
    genes = [f"Gene{i:04d}" for i in range(N_GENES)]
    adata = ad.AnnData(X=X, obs={"batch": np.array([f"b{seed}"] * n_cells)})
    adata.var_names = genes
    adata.obs_names = [f"c{i:05d}" for i in range(n_cells)]
    return adata


make_adata(1500, 1, True).write_h5ad(DATA_DIR / "sample_a.h5ad")
make_adata(800, 2, False).write_h5ad(DATA_DIR / "sample_b.h5ad")

raw_paths = sorted(DATA_DIR.glob("*.h5ad"))
raw_paths

## Step 2 — Convert to `.zarr.zip`

`AnnDataZarrZipConverter` is a callable dataclass: given any AnnData-readable
source, it writes a same-name `.zarr.zip` and returns that path. Notable
options:

| option | meaning |
|---|---|
| `format="auto"` | sparse `X` → CSR store, dense `X` → `dense1d` (cell-aligned) |
| `align_cells=True` | chunk grid is aligned to whole cells — best for cell-wise access |
| `chunk_size` | cells per chunk (or a `[cells, genes]` tuple) |
| `layer_format` | how `adata.layers` are written (`"auto"` / `"preserve"` / `"sparse"` / `"dense1d"` / `"dense2d"`) |
| `output_dir`, `overwrite` | where to write and whether to clobber |

Set `convert_sources = False` to skip conversion when your inputs are already
scdata stores.

In [ ]:
convert_sources = True

if convert_sources:
    converter = AnnDataZarrZipConverter(
        chunk_size=100_000,
        align_cells=True,
        layer_format="auto",
    )
    store_paths = [converter(path) for path in raw_paths]
else:
    store_paths = [Path(p) for p in raw_paths]

store_paths

## Step 3 — Register all datasets

`launch(path)` **parses metadata only** — it reads zarr `zarr.json`, resolves
chunk locations, and returns a `Dataset` (`DenseDataset` or `SparseDataset`).
Numeric chunks are not decoded here. `bank.register(...)` accepts a single
dataset or any iterable of them and returns one `DatasetId` per store (in order).

The bank is configured through `DataBankConfig`. It is a plain dataclass tree;
`.make(**kwargs)` routes flat kwargs to the right nested field so you rarely
build sub-configs by hand. The defaults are a sensible thread-pool setup; below
we bump the decode pool and the chunk cache for illustration.

In [ ]:
config = DataBankConfig.make(
    backend="threaded",                # -> io_config.backend ("uring" on Linux w/ io_uring)
    decode__num_workers=4,             # -> decode_config.num_workers
    cache_capacity_bytes=128 * 1024**2,  # -> access_config.cache_capacity_bytes
)

bank = ScDataBank(config)

# Key line: path list -> launched datasets -> list[DatasetId]
dataset_ids = bank.register(map(launch, store_paths))

for file_id, did in enumerate(dataset_ids):
    print(
        file_id,
        did,
        "cells=", bank.dataset_num_cells(did),
        "genes=", bank.dataset_num_genes(did),
        "dtype=", bank.dataset_dtype(did),
    )

## Step 4 — Build the `(file_id, cell_id)` dataset

`ScDataLoader` keeps torch's usual sampling / shuffle / batch-size / drop-last
behavior, but each sample must identify a cell as **two integers**: which
registered file and which cell within that file. The small `CellIndexDataset`
below maps a flat global index to such a pair via a prefix-sum (`bisect`),
without ever materializing the full list of all pairs — important when the
corpus is tens of millions of cells.

Any map-style object returning `(file_id, cell_id)` works; you can also return
a `dict` with `"file_id"` / `"cell_id"` keys, or a 2-element list/array.

In [ ]:
class CellIndexDataset(Sequence[tuple[int, int]]):
    """Flat index -> (file_id, cell_id), via a prefix-sum over cells per file.

    A :class:`collections.abc.Sequence` so it satisfies the
    ``Sequence[...]`` type of ``ScDataLoader``'s ``dataset`` argument; the
    ``__getitem__`` / ``__len__`` torch map-style protocol is unchanged.
    """

    def __init__(self, cells_per_file):
        self.offsets = [0]
        for count in cells_per_file:
            self.offsets.append(self.offsets[-1] + int(count))

    def __len__(self) -> int:
        return self.offsets[-1]

    def __getitem__(self, index: int) -> tuple[int, int]:
        if index < 0:
            index += len(self)
        if index < 0 or index >= len(self):
            raise IndexError(index)
        file_id = bisect_right(self.offsets, index) - 1
        cell_id = index - self.offsets[file_id]
        return file_id, cell_id


cells_per_file = [bank.dataset_num_cells(did) for did in dataset_ids]
cell_dataset = CellIndexDataset(cells_per_file)

len(cell_dataset), cell_dataset[0], cell_dataset[len(cell_dataset) - 1]

## Step 5 — Collate decoded `CellBatch` objects

`ScDataLoader` does **not** call the default torch collate. On `iter(loader)` it
materializes the torch batch indices, groups each batch by `file_id`, starts one
`bank.prefetch(...)` stream per file, and yields a **plain dict** to your
`collate_fn`:

| key | value |
|---|---|
| `file_ids` | `intp` array, file id per row, in original batch order |
| `cell_ids` | `intp` array, cell id per row, in original batch order |
| `batches` | `{file_id: CellBatch}` — already decoded by the bank |
| `positions` | `{file_id: intp array}` — where those rows go in the mixed batch |
| `cells` | `{file_id: intp array}` — the requested cell ids for that file |

When a batch contains a single file id, convenience keys `file_id`, `batch`,
`cells_in_file`, `positions_in_batch` are also present. A `CellBatch` exposes
`to_numpy()` (2D `[num_cells, num_genes]` zero-copy view), `matrix`, `num_genes`,
and `var_names` (gene names when known).

The collate below stitches the per-file decoded blocks back into one dense
`[batch_size, num_genes]` tensor using `positions`, and threads source ids
through for tracing. It assumes every file in a mixed batch shares the same
column layout — which is guaranteed when `genes` is passed (all files project to
that same gene list) or when all stores share gene order.

In [ ]:
def collate_scdata_batch(batch):
    first = next(iter(batch["batches"].values()))
    num_genes = first.num_genes
    x = np.empty((len(batch["cell_ids"]), num_genes), dtype=first.data.dtype)

    for file_id, cell_batch in batch["batches"].items():
        positions = batch["positions"][file_id]
        x[positions] = cell_batch.to_numpy()

    return {
        "x": torch.from_numpy(x),
        "file_ids": torch.as_tensor(batch["file_ids"], dtype=torch.long),
        "cell_ids": torch.as_tensor(batch["cell_ids"], dtype=torch.long),
        "gene_names": first.var_names,
    }

## Step 6 — Create the loader

Two things to decide here:

* **Gene projection.** Pass `genes=[...]` to decode only those columns, in that
  order, for every file. With `missing="zero"`, genes absent from a given store
  are filled with zeros instead of raising — this is how batches can mix
  datasets with non-identical gene sets. Leave `genes=None` to decode every gene
  in dataset order (then all stores must share that order for mixed batches).
* **Prefetch tuning.** `ScheduledPrefetchConfig` controls how far ahead the bank
  decodes (`prefetch_step`) and how many decode stages run ahead
  (`access__decode_ahead_steps`). The defaults are fine for most cases.

`num_workers` stays `0`: `ScDataLoader` overrides `__iter__`, so the prefetch
pipeline runs in the main process driven by the bank's own Rust thread pools —
torch worker processes are not used for decoding.

In [ ]:
# Project every batch onto the first 50 genes — mixed-file batches then share
# an exact column layout even if the stores differ elsewhere.
genes = bank.dataset_genes(dataset_ids[0])[:50]
# genes = None  # decode every gene in dataset order

prefetch_config = ScheduledPrefetchConfig.make(
    prefetch_step=2,
    access__decode_ahead_steps=1,
)

loader = ScDataLoader(
    bank,
    cell_dataset,
    batch_size=1024,
    shuffle=True,
    drop_last=False,
    num_workers=0,
    dataset_ids=dataset_ids,
    genes=genes,
    missing="zero",
    prefetch_config=prefetch_config,
    collate_fn=collate_scdata_batch,
)

## Step 7 — Consume batches

From here on, downstream code sees normal torch tensors plus source ids for
tracing. Each batch is a dense `[batch_size, num_genes]` float matrix (here
`num_genes=50` because of the projection above), with `file_ids` / `cell_ids`
to map any row back to its origin cell.

In [ ]:
for step, batch in enumerate(loader):
    x = batch["x"].float()
    file_ids = batch["file_ids"]
    cell_ids = batch["cell_ids"]

    print(
        step,
        x.shape,
        "files=", torch.unique(file_ids).tolist(),
        "first5=", file_ids[:5].tolist(),
        "genes=", len(batch["gene_names"]),
    )

    # model input example:
    # logits = model(x)
    # loss = loss_fn(logits, targets)
    # loss.backward()

    if step == 2:
        break

## Step 8 — Direct random access

The same registered ids work for direct, synchronous access via `bank.load(...)`,
which is useful for debugging, validation, or building small held-out probes.
`load` accepts an optional `genes` subset (same `missing` policy) and an optional
`dtype` to cast the decoded payload — short names like `"f32"`, `"f16"`, `"u8"`
or a `DType` enum are accepted.

In [ ]:
# Three cells from dataset 0, projected onto the same 50 genes, cast to f32.
preview = bank.load(dataset_ids[0], [0, 1, 2], genes=genes, missing="zero", dtype="f32")
print("projected:", preview.to_numpy().shape, preview.var_names[:3])

# Two cells from dataset 1 with ALL genes (no projection), cast to f32.
preview_full = bank.load(dataset_ids[1], [0, 1], dtype="f32")
print("full genes:", preview_full.to_numpy().shape, "num_genes=", preview_full.num_genes)

## Step 9 — Lifecycle

`ScDataBank` owns Rust IO / decode / access / compute thread pools. Release them
deterministically when you are done:

* prefer the **context-manager** form (`with ScDataBank(...) as bank:`) so pools
  are torn down on scope exit even if an exception fires;
* otherwise `unregister(ids)` / `unregister_all(ids)` to drop datasets, then
  `bank.close()` to release the core. `close()` is idempotent; any later call
  on a closed bank raises.

In [ ]:
# Recommended: scope the bank so its pools are torn down automatically.
with ScDataBank() as ctx_bank:
    did = ctx_bank.register(launch(store_paths[0]))
    print("ctx bank registered:", did, "cells=", ctx_bank.dataset_num_cells(did))
    # ... use ctx_bank ...
# pools released here

# Tear down the long-lived bank from the earlier steps.
bank.unregister_all(dataset_ids)
bank.close()
print("bank closed:", bank.is_closed)

## Recap

* `launch` parses a store; `ScDataBank.register` turns parsed datasets into
  opaque `DatasetId` handles the Rust core can decode.
* Your torch dataset only emits `(file_id, cell_id)` pairs — `ScDataLoader`
  groups each batch by file and streams decoded `CellBatch` blocks from the
  bank into your `collate_fn`.
* Pass a `genes` list (plus `missing="zero"`) to mix datasets with differing
  gene sets safely; leave it `None` when all stores share gene order.
* Tune IO / decode / cache via `DataBankConfig.make(...)`, and prefetch depth
  via `ScheduledPrefetchConfig`.
* Use the context-manager form to release Rust pools deterministically.

See `docs/benchmark-report.md` for throughput numbers and the configuration
knobs that move them, and the `scdata` package for the full API surface
(`launch_all` / `register_all` for multi-layer stores, `CellAccess` /
`CellData` for lower-level access, `IoConfig.uring(...)` for the io_uring
backend).